# Tutorial 04 - Exercise 02: Student Grades Analysis with NumPy

Analyses a synthetic dataset of 100 students across 5 academic modules
(`THL`, `DSS`, `DB`, `SE`, `RE`).


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/05_NumPy_Scientific_Computing/` project folder (anchored on its own notebook path) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "05_NumPy_Scientific_Computing"
ANCHOR_REL = "notebooks/02_student_grades_analysis.ipynb"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate the Dataset


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(101)
grades = np.random.randint(0, 21, size=(100, 5))
modules = ["THL", "DSS", "DB", "SE", "RE"]
print(f"Dataset shape: {grades.shape}")

# Name given to the same array in the tutorial handout
grades_of_students = grades

### 2. Task 1: Mean Grade per Student (`mean_per_student`)

`axis=1` reduces each row, so one number is produced per student. The handout calls the array
`grades_of_students`; that name is an alias of `grades`, not a second dataset. A row-by-row
recomputation with plain Python is printed next to the vectorised result: the two must agree.

In [ ]:
mean_per_student = grades_of_students.mean(axis=1)

print("shape:", mean_per_student.shape)
print("first five means:", np.round(mean_per_student[:5], 2).tolist())

# Independent recomputation (row by row) must give the same vector
recomputed = np.array([sum(row) / row.size for row in grades_of_students])
print("matches the row-by-row recomputation:", bool(np.allclose(mean_per_student, recomputed)))
print(f"cohort mean: {mean_per_student.mean():.2f} | best student: {mean_per_student.max():.2f} | weakest: {mean_per_student.min():.2f}")

### 3. Summary Statistics per Module

In [ ]:
for i, m in enumerate(modules):
    col = grades[:, i]
    print(f"Module {m:4s}: Mean={col.mean():.2f} | Median={np.median(col):.1f} | Std={col.std():.2f} | Var={col.var():.2f}")

print(f"\nOverall Min: {grades.min()} | Overall Max: {grades.max()}")


### 4. THL Module Percentiles

In [ ]:
thl = grades[:, 0]
p25, p50, p75 = np.percentile(thl, [25, 50, 75])
print(f"THL Percentiles -> 25th: {p25:.1f}, Median: {p50:.1f}, 75th: {p75:.1f}")


### 5. Boolean Masking: Scores Between 10 and 15

In [ ]:
mask = (grades >= 10) & (grades <= 15)
filtered = grades[mask]
print(f"Grades in range [10, 15]: {filtered.size} occurrences")


### 6. Distributions and Comparison Visualizations
The boxplot uses the `tick_labels` parameter (Matplotlib >= 3.9) with a
fallback to the legacy `labels` keyword on older versions.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Plot A: Average Grades
axs[0].bar(modules, grades.mean(axis=0), color="skyblue", edgecolor="black")
axs[0].axhline(10, color="red", linestyle="--", label="Passing Mark")
axs[0].set_title("Module Average Grades", fontweight="bold")
axs[0].set_ylim(0, 20)
axs[0].legend()

# Plot B: Grade Spread Boxplot (Matplotlib 3.9+ renamed labels -> tick_labels)
box_kwargs = dict(patch_artist=True, boxprops=dict(facecolor="lightgreen"))
try:
    axs[1].boxplot(grades, tick_labels=modules, **box_kwargs)
except TypeError:
    axs[1].boxplot(grades, labels=modules, **box_kwargs)
axs[1].set_title("Module Grade Distributions", fontweight="bold")

plt.tight_layout()
plt.show()


### 7. Extra Practice

**Task E1 - students whose grades are all above 15.** The handout says *greater than 15*. The vault
tutorial note (`03.3`) states the same exercise with *15 or more*. Both masks are built so the
boundary is visible: a grade of exactly 15 is excluded by the first mask and admitted by the second.
On this dataset `0` students satisfy `> 15` and `0` satisfy `>= 15`
(`0` students sit exactly on the boundary).

In [ ]:
all_above_15 = (grades_of_students > 15).all(axis=1)
all_at_least_15 = (grades_of_students >= 15).all(axis=1)

print("students with every grade > 15 :", int(all_above_15.sum()))
print("students with every grade >= 15:", int(all_at_least_15.sum()))
print("admitted only by the >= version :", int((all_at_least_15 & ~all_above_15).sum()))
print("student numbers (> 15):", (np.where(all_above_15)[0] + 1).tolist())
print("rows with a grade of exactly 15 :", int((grades_of_students == 15).any(axis=1).sum()))

**Task E2 - top 5 students by average, with `np.argsort`.** `argsort` returns the positions that
would sort the vector, not the values; negating the means turns the ascending sort into a
descending one and `kind="stable"` keeps equal means in student order. Expected ranking:
#25 (17.00), #30 (16.80), #4 (15.60), #100 (15.60), #40 (15.40).

In [ ]:
order = np.argsort(-mean_per_student, kind="stable")
top5 = order[:5]

for rank, student in enumerate(top5, start=1):
    print(f"rank {rank}: student {student + 1:3d} | mean {mean_per_student[student]:.2f} | grades {grades_of_students[student].tolist()}")
print("top-5 means:", np.round(mean_per_student[top5], 2).tolist())
print("same answer with a plain sort:", bool(mean_per_student[top5].tolist() == sorted(mean_per_student, reverse=True)[:5]))

**Task E3 - the module with the highest average grade.** Reducing with `axis=0` gives one mean per
module; `argmax` then returns the position of the largest. Module means (THL, DSS, DB, SE, RE):
THL 9.45, DSS 10.24, DB 10.32, SE 10.58, RE 10.50 - the highest is
module index 3.

In [ ]:
module_means = grades_of_students.mean(axis=0)
best_module = int(np.argmax(module_means))

for name, value in zip(modules, module_means):
    print(f"{name}: {value:.2f}")
print(f"highest average: {modules[best_module]} ({module_means[best_module]:.2f})")
print("lowest average :", modules[int(np.argmin(module_means))], f"({module_means.min():.2f})")
print("matches the mean of the per-student column:", bool(np.isclose(module_means.mean(), mean_per_student.mean())))